# GTSRB Traffic-Sign Recognition — Colab GPU Run

This notebook is configured for a Google Colab GPU. It downloads and verifies the complete GTSRB classification dataset, creates EDA graphs, trains a custom CNN and MobileNetV2 with a longer GPU budget, evaluates both models, creates Grad-CAM explanations, and exports report-ready artifacts.

**Before running:** Colab menu → Runtime → Change runtime type → select **T4/L4 GPU**. Run cells from top to bottom. If the repository is not already in `/content/mait-mini`, upload a ZIP of this project when prompted.

In [ ]:
# Install project dependencies. Colab supplies TensorFlow and its GPU runtime.
%pip -q install opencv-python-headless pandas matplotlib scikit-learn Pillow pytest

import os, sys, json, shutil, zipfile, subprocess
from pathlib import Path
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)
assert gpus, 'GPU not detected. Change the Colab runtime to T4/L4 GPU and rerun.'
for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
print('GPU runtime is ready.')

In [ ]:
# Everything is stored directly in the temporary Colab runtime. Upload a project ZIP if needed.
from google.colab import files
PROJECT_DIR = Path('/content/mait-mini')
if not (PROJECT_DIR / 'scripts/download_dataset.py').exists():
    print('Upload a ZIP of the mait-mini project, including src/, scripts/, and requirements.txt.')
    uploaded = files.upload()
    archive = next(iter(uploaded))
    with zipfile.ZipFile(archive) as z: z.extractall('/content')
    assert (PROJECT_DIR / 'scripts/download_dataset.py').exists(), 'Expected project at /content/mait-mini after upload.'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
os.environ['PYTHONPATH'] = str(PROJECT_DIR / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project:', PROJECT_DIR)
print('Dataset and artifacts will be saved locally under:', PROJECT_DIR)

In [ ]:
# Download the full dataset directly from Zenodo and verify its published MD5 before extraction.
subprocess.run([sys.executable, 'scripts/download_dataset.py'], cwd=PROJECT_DIR, check=True)
# This opens a direct browser download of the 320 MB archive to your computer.
files.download(str(PROJECT_DIR / 'data/raw/gtsrb/data.zip'))

In [ ]:
# EDA: class balance and representative images.
subprocess.run([sys.executable, 'scripts/run_eda.py'], cwd=PROJECT_DIR, check=True)
from traffic_sign_recognition.data import load_labeled_data
train_df, test_df = load_labeled_data(PROJECT_DIR / 'data/raw/gtsrb')
print(f'train={len(train_df):,}; official_test={len(test_df):,}; classes={train_df.label.nunique()}')
from IPython.display import display, Image
display(Image(filename='artifacts/eda/class_distribution.png', width=900))
display(Image(filename='artifacts/eda/sample_grid.png', width=700))

## GPU training configuration

These settings are stronger than the local baseline. `--require-gpu` makes each run stop if Colab silently falls back to CPU. Class balancing is enabled so macro F1 remains meaningful for the uneven 43-class distribution.

In [ ]:
BATCH_SIZE = 128
CUSTOM_EPOCHS = 30
MOBILENET_EPOCHS = 25
SEED = 42
BALANCED = True

def train_model(name, epochs):
    cmd = [sys.executable, '-m', 'traffic_sign_recognition.train', '--model', name, '--epochs', str(epochs), '--batch-size', str(BATCH_SIZE), '--seed', str(SEED), '--require-gpu', '--output-dir', f'artifacts/models/{name}']
    if BALANCED: cmd.append('--balanced')
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
train_model('custom_cnn', CUSTOM_EPOCHS)
train_model('mobilenet_v2', MOBILENET_EPOCHS)

In [ ]:
# Evaluate on the untouched official test set and rebuild dashboard-ready results.
for name in ('custom_cnn', 'mobilenet_v2'):
    subprocess.run([sys.executable, '-m', 'traffic_sign_recognition.evaluate', '--model-path', f'artifacts/models/{name}/model.keras'], cwd=PROJECT_DIR, check=True)
    subprocess.run([sys.executable, 'scripts/error_analysis.py', '--evaluation-dir', f'artifacts/models/{name}/evaluation'], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, 'scripts/build_results.py'], cwd=PROJECT_DIR, check=True)
display(Image(filename='site/dist/assets/model_comparison.png', width=800))
for name in ('custom_cnn', 'mobilenet_v2'): display(Image(filename=f'artifacts/models/{name}/evaluation/confusion_matrix.png', width=650))

In [ ]:
# Print report-ready metrics and hardware evidence.
results = json.loads((PROJECT_DIR / 'site/dist/data/results.json').read_text())
for row in results['models']:
    folder = PROJECT_DIR / 'artifacts/models' / row['name']
    config = json.loads((folder / 'config.json').read_text())
    report = json.loads((folder / 'evaluation/classification_report.json').read_text())
    history = json.loads((folder / 'history.json').read_text())
    print(row['name'], {'accuracy': report['accuracy'], 'macro_precision': report['macro avg']['precision'], 'macro_recall': report['macro avg']['recall'], 'macro_f1': report['macro avg']['f1-score'], 'training_device': config.get('training_device'), 'epochs_completed': len(history['loss'])})

In [ ]:
# Grad-CAM explanation for a MobileNetV2 prediction.
sample = str(test_df.iloc[0].path)
subprocess.run([sys.executable, 'scripts/generate_gradcam.py', '--model-path', 'artifacts/models/mobilenet_v2/model.keras', '--image', sample, '--output', 'artifacts/explanations/colab_gradcam.png'], cwd=PROJECT_DIR, check=True)
display(Image(filename='artifacts/explanations/colab_gradcam.png', width=500))

In [ ]:
# Download results directly to your computer. Colab storage is temporary, so run this after training.
result_files = ['site/dist/data/results.json', 'site/dist/assets/model_comparison.png', 'artifacts/eda', 'artifacts/models', 'artifacts/explanations']
local_bundle = Path('/content/gtsrb_colab_results')
local_bundle.mkdir(exist_ok=True)
for source in result_files:
    src, dst = PROJECT_DIR / source, local_bundle / Path(source).name
    if src.is_dir(): shutil.copytree(src, dst, dirs_exist_ok=True)
    elif src.exists(): shutil.copy2(src, dst)
archive = shutil.make_archive('/content/gtsrb_colab_results', 'zip', local_bundle)
print('Saved locally:', archive)
files.download(archive)

In [ ]:
# Download the completed notebook directly as well.
notebook_candidates = list(Path('/content').glob('*.ipynb')) + list(Path('/content/mait-mini').glob('*.ipynb'))
if notebook_candidates:
    print('Downloading:', notebook_candidates[0])
    files.download(str(notebook_candidates[0]))
else:
    print('Use Colab File → Download → Download .ipynb to save the completed notebook.')

## What to send back

Send the downloaded `gtsrb_colab_results.zip` and completed `.ipynb` directly. No Google Drive is used. The ZIP contains `results.json`, classification reports, confusion matrices, learning curves, trained models, EDA graphs, and the Grad-CAM image needed to update the project documents.